# Práctica: Regresión lineal multivariada con y sin regularización (Diabetes)

Objetivo: Predecir el progreso de la diabetes usando el dataset load_diabetes de scikit-learn.

LinearRegression (sin regularización) vs Ridge (L2) vs Lasso (L1)

Se evalúa en test con: MSE, R2

Se probarán varios valores de alpha en Ridge y Lasso y se comparan.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score

## 1) Carga y exploración inicial
- Cargamos el dataset Diabetes.
- Revisamos dimensiones y tipos (todas numéricas).
- Exploratorio rápido: describe y correlaciones simples con y.

In [ ]:
data = load_diabetes()
X = data.data
y = data.target
feature_names = data.feature_names

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Nombres de variables:", feature_names)

df = pd.DataFrame(X, columns=feature_names)
df["target"] = y
display(df.head())
display(df.describe())

In [ ]:
corrs = df.corr(numeric_only=True)["target"].drop("target").sort_values(ascending=False)
print("Correlaciones con target (ordenadas):")
display(corrs.to_frame("corr_with_target"))

plt.figure()
plt.bar(corrs.index, corrs.values)
plt.xticks(rotation=90)
plt.title("Correlación de cada variable con target (Diabetes)")
plt.ylabel("correlación")
plt.tight_layout()
plt.show()

## 2) División entrenamiento / test
- 80% train, 20% test
- random_state=11 (reproducible).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=11
)

print("Train:", X_train.shape, "Test:", X_test.shape)

## 3) Entrenamiento de modelos
Modelos:
LinearRegression(), Ridge(alpha=...) y Lasso(alpha=...)

In [ ]:
lin = LinearRegression()
lin.fit(X_train, y_train)

# valores de alpha a probar
alphas = [0.01, 0.1, 1, 10, 100]

ridge_models = {}
lasso_models = {}

for a in alphas:
    r = Ridge(alpha=a)
    r.fit(X_train, y_train)
    ridge_models[a] = r

    # Subimos max_iter para evitar problemas de convergencia
    l = Lasso(alpha=a, max_iter=50000)
    l.fit(X_train, y_train)
    lasso_models[a] = l

## 4) Evaluación (MSE y R2 en test)
- Para cada modelo: predecimos en X_test
- Calculamos MSE y R2
- Tabla resumen

In [ ]:
rows = []

# LinearRegression
y_pred = lin.predict(X_test)
rows.append({
    "model": "LinearRegression",
    "alpha": np.nan,
    "MSE": mean_squared_error(y_test, y_pred),
    "R2": r2_score(y_test, y_pred)
})

# Ridge
for a, model in ridge_models.items():
    y_pred = model.predict(X_test)
    rows.append({
        "model": "Ridge",
        "alpha": a,
        "MSE": mean_squared_error(y_test, y_pred),
        "R2": r2_score(y_test, y_pred)
    })

# Lasso
for a, model in lasso_models.items():
    y_pred = model.predict(X_test)
    rows.append({
        "model": "Lasso",
        "alpha": a,
        "MSE": mean_squared_error(y_test, y_pred),
        "R2": r2_score(y_test, y_pred)
    })

df_results = pd.DataFrame(rows)

display(df_results.sort_values(["model", "alpha"]))

In [ ]:
best_by_mse = df_results.sort_values("MSE").iloc[0]
print("Mejor por MSE:")
display(best_by_mse.to_frame().T)

best_by_r2 = df_results.sort_values("R2", ascending=False).iloc[0]
print("Mejor por R2:")
display(best_by_r2.to_frame().T)

## 5) Gráficas simples de comparación

- MSE vs alpha para Ridge y Lasso
- R2 vs alpha para Ridge y Lasso
Incluimos una línea horizontal con el valor de LinearRegression para comparar.

In [ ]:
lin_mse = df_results[df_results["model"] == "LinearRegression"]["MSE"].values[0]
lin_r2 = df_results[df_results["model"] == "LinearRegression"]["R2"].values[0]

ridge_df = df_results[df_results["model"] == "Ridge"].sort_values("alpha")
lasso_df = df_results[df_results["model"] == "Lasso"].sort_values("alpha")

plt.figure()
plt.plot(ridge_df["alpha"], ridge_df["MSE"], marker="o", label="Ridge")
plt.plot(lasso_df["alpha"], lasso_df["MSE"], marker="o", label="Lasso")
plt.axhline(lin_mse, linestyle="--", label="LinearRegression")
plt.xscale("log")
plt.title("MSE en test vs alpha")
plt.xlabel("alpha (log)")
plt.ylabel("MSE")
plt.legend()
plt.show()

plt.figure()
plt.plot(ridge_df["alpha"], ridge_df["R2"], marker="o", label="Ridge")
plt.plot(lasso_df["alpha"], lasso_df["R2"], marker="o", label="Lasso")
plt.axhline(lin_r2, linestyle="--", label="LinearRegression")
plt.xscale("log")
plt.title("R2 en test vs alpha")
plt.xlabel("alpha (log)")
plt.ylabel("R2")
plt.legend()
plt.show()

## 6) Coeficientes y efecto de Lasso (coeficientes a cero)

- Comparamos coeficientes de LinearRegression, Ridge(alpha=1) y Lasso(alpha=1) como ejemplo.
- Contamos cuántos coeficientes quedan en cero en Lasso (para cada alpha).

In [ ]:
coef_table = pd.DataFrame({
    "feature": feature_names,
    "LinearRegression": lin.coef_
})

coef_table["Ridge_alpha_1"] = ridge_models[1].coef_
coef_table["Lasso_alpha_1"] = lasso_models[1].coef_

display(coef_table)

zero_counts = []
for a, model in lasso_models.items():
    zeros = int(np.sum(model.coef_ == 0))
    zero_counts.append({"alpha": a, "n_zeros": zeros})

df_zeros = pd.DataFrame(zero_counts).sort_values("alpha")
print("Número de coeficientes exactamente cero en Lasso:")
display(df_zeros)

plt.figure()
plt.plot(df_zeros["alpha"], df_zeros["n_zeros"], marker="o")
plt.xscale("log")
plt.title("Lasso: número de coeficientes en cero vs alpha")
plt.xlabel("alpha (log)")
plt.ylabel("nº coeficientes = 0")
plt.show()

## 7) Comentarios - Resumen

Qué modelo funciona mejor:
En mis resultados, el mejor modelo en test es LinearRegression, con MSE = 2935.81 y R2 = 0.3982.

Comparación con Ridge (L2):
Ridge no mejora respecto a la regresión lineal. Con alpha pequeños se queda cerca pero es ligeramente peor:
- alpha=0.01: MSE = 2967.71, R2 = 0.3917
- alpha=0.1:  MSE = 2956.70, R2 = 0.3939
A medida que sube alpha el modelo se vuelve más rígido y el rendimiento cae bastante; por ejemplo, con alpha=10 el R2 baja a 0.1338. En este split, la regularización L2 no aporta una mejora clara.

Comparación con Lasso (L1):
En Lasso ocurre algo parecido: el mejor valor probado es alpha=0.1, con MSE = 2943.32 y R2 = 0.3967, muy cerca de LinearRegression pero sin superarlo. Con alpha=1 empeora, y con alpha=10 o 100 la penalización es tan fuerte que el modelo pierde demasiada información y el R2 llega a ser negativo.

Coeficientes en cero (Lasso):
En Lasso se ve claramente el efecto de selección de variables. Al aumentar alpha crece el número de coeficientes exactamente cero:
- alpha=1: 7 coeficientes a cero
- alpha=10 y alpha=100: 10 coeficientes a cero
Esto ocurre porque la penalización L1 fuerza a eliminar variables menos útiles y quedarse con las más relevantes.

Conclusión:
En este dataset y con este split (80/20), la regularización no mejora el rendimiento frente a LinearRegression. Aun así, Lasso es útil para interpretar el modelo porque selecciona variables al hacer que varios coeficientes se vuelvan cero.